In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [2]:
# latencies: 50, 90 150, 210
default_region = ['us-west1-b']
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

regions = ['us-west1-b', 'us-west1-b', 'us-west1-b', 'us-west1-b']


# Regions

# num_nodes = 4
zone_no = 0
for num_nodes in  [4]:
# for zone_no in  [0,1,2,3, 4]:


    project = "research-488322"
    zone = "us-central1-c"
    machine_type = "e2-standard-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

    

    # Create commands list
    commands = []
    
    for i in range(num_nodes):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
    
    
    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)
    
    
    # #Parallel instance creation
    
    # with concurrent.futures.ThreadPoolExecutor(max_workers=48) as executor:
    #     futures = [executor.submit(run_command, cmd) for cmd in commands]
    #     concurrent.futures.wait(futures)
    
    print("All instances launched.")
    


➡ Existing instances to delete:

✔ No tsm-sc-* instances found.

All instances launched.


In [3]:
    # # Wait a bit for IPs to propagate
    # import time
    # # time.sleep(30)
    

    # # Get IPs
    # os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
    #           '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')
    
    # with open('tsm_ips.txt', 'r') as f:
    #     iplist = [line.strip() for line in f.readlines()]
    
    # print("🎯 Instance IPs:", iplist)
    

In [4]:
    os.system('git add .; git commit -m "testing"; git push')
   

    n_collection = 10
    os.system('make -j8')
    
    

[main 537298d] testing
 1 file changed, 1272 insertions(+), 316 deletions(-)


To github.com:tejas-shivanand-mane/stellar-core.git
   ca8efe8..537298d  main -> main


make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/contrib'
Making all in dist-build
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
Making all in msvc-scripts
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/msvc-

soci/src/backends/postgresql/standard-use-type.cpp: In member function ‘virtual void soci::postgresql_standard_use_type_backend::pre_use(const soci::indicator*)’:
soci/src/backends/postgresql/standard-use-type.cpp:135:45: warning: ‘%02d’ directive output may be truncated writing between 2 and 11 bytes into a region of size between 8 and 18 [-Wformat-truncation=]
  135 |                 snprintf(buf_, bufSize, "%d-%02d-%02d %02d:%02d:%02d",
      |                                             ^~~~
soci/src/backends/postgresql/standard-use-type.cpp:135:41: note: directive argument in the range [-2147483647, 2147483647]
  135 |                 snprintf(buf_, bufSize, "%d-%02d-%02d %02d:%02d:%02d",
      |                                         ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~
In file included from /usr/include/stdio.h:974,
                 from /usr/include/c++/15/cstdio:47,
                 from /usr/include/c++/15/ext/string_conversions.h:47,
                 from /usr/include/c++/15/bits/

ranlib libmedida.a
rm -f lib3rdparty.a
ar cr lib3rdparty.a util/siphash.o util/cbitset.o util/getopt_long.o util/crc16.o http/request_parser.o http/connection.o http/HttpClient.o http/connection_manager.o http/server.o http/reply.o httpthreaded/request_parser.o httpthreaded/connection.o httpthreaded/server.o httpthreaded/reply.o asio.o json/jsoncpp.o sqlite/shell.o sqlite/sqlite3.o sqlite/carray.o spdlog.o 
ranlib lib3rdparty.a


...

depbase=`echo invariant/test/InvariantTests.o | sed 's|[^/]*$|.deps/&|;s|\.o$||'`;\
g++ -std=c++17 -DHAVE_CONFIG_H -I. -I..  -isystem ".." -I"../src" -I"../src" -isystem /home/tejas/stellar-core/lib/libsodium/src/libsodium/include -isystem ../lib/libsodium/src/libsodium/include -isystem ../lib/libsodium/src/libsodium/include/sodium -isystem /home/tejas/stellar-core/lib/xdrpp -isystem /home/tejas/stellar-core/lib/xdrpp -isystem ../lib/libmedida/src -isystem ../lib/soci/src/core -isystem ../lib/sqlite -DSQLITE_CORE -DSQLITE_OMIT_LOAD_EXTENSION=1 -DASIO_SEPARATE_COMPILATION=1 -DASIO_STANDALONE -isystem ../lib/asio/asio/include -I/usr/include/x86_64-linux-gnu  -isystem "../lib" -isystem "../lib/autocheck/include" -isystem "../lib/cereal/include" -isystem "../lib/util" -isystem "../lib/fmt/include" -isystem "../lib/soci/src/core" -isystem "../lib/tracy/public/tracy" -isystem "../lib/spdlog/include" -isystem "../rust/src" -DUSE_POSTGRES=1 -I/usr/include/postgresql   -I"../src/protocol-curr" 

At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics
At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics
At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: un

make[3]: Leaving directory '/home/tejas/stellar-core/src'
make[2]: Leaving directory '/home/tejas/stellar-core/src'
make[1]: Leaving directory '/home/tejas/stellar-core'


make[2]: *** [Makefile:2849: all] Error 2
make[1]: *** [Makefile:713: all-recursive] Error 1
make: *** [Makefile:526: all] Error 2


512

In [ ]:
    def kill_stellar_private(i):


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill -9 stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    
    
    
    def git_pull_stellar(i):
        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    git pull"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
    print(results)
    

In [4]:
    import shutil
    
    if os.path.exists('../stellar-private'):
        
        shutil.rmtree('../stellar-private')
    os.mkdir('../stellar-private')
    
    
    os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')
    
    os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')
    
    # --- Configuration ---
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg" 
    
    # --- The os.system() Command ---
    # This command prepends the line to the target_file on your local machine.
    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')

    target_file = "../stellar-private/node2/stellar-core.cfg" 
    line_to_add = "MEMORY_PROF=true"

    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')
    
    print(f"The line '{line_to_add}' has been prepended to {target_file}.")
    

Detected 4 nodes based on tsm_ips.txt.
Cleaning and creating directory for node1. Ports: Peer 11625, HTTP 11626...
Cleaning and creating directory for node2. Ports: Peer 11635, HTTP 11636...
Cleaning and creating directory for node3. Ports: Peer 11645, HTTP 11646...
Cleaning and creating directory for node4. Ports: Peer 11655, HTTP 11656...
Generating seed for node1...
Generating seed for node2...
Generating seed for node3...
Generating seed for node4...
Creating config file for node1...
Creating config file for node2...
Creating config file for node3...
Creating config file for node4...
Detected 4 nodes based on tsm_ips.txt.
Initializing database for node1...
Initializing database for node2...
Initializing database for node3...
Initializing database for node4...


2026-06-12T04:36:26.098 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2026-06-12T04:36:26.100 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "GAU4E", "node4", "node2", "node3" ]
}

2026-06-12T04:36:26.100 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-12T04:36:26.100 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2026-06-12T04:36:26.130 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2026-06-12T04:36:26.132 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node1", "node4", "GDHN5", "node3" ]
}

2026-06-12T04:36:26.132 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-12T04:36:26.132 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2026-06-12T04:36:26.161 [default INFO] Config from /home/tejas/stellar-private/node3/ste

✅ 4-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &
The line 'MEMORY_PROF=true' has been prepended to ../stellar-private/node2/stellar-core.cfg.


In [ ]:
    def compile_stellar(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    make -j16; cd; sudo rm -r stellar-private"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(compile_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    def clean_stellar_private(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]

        
        remote_command = f"""\
    cd /home/tejas; \
    sudo rm -r stellar-private; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))
    

    def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
        """
        Constructs and executes the gcloud compute scp command to copy a folder
        to a specific GCP instance.
        """


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        instance_name = f"tsm-sc-{i:03}"
        
        # The --recurse flag is crucial for copying folders
        # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{source_folder}" "{instance_name}:{destination_path}"'
    
        print(f"Executing command for {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Command for {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    
    results = Parallel(n_jobs=48)(
        delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
    )
    
    

    
    # def setup_stellar_private(i):
    #     command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    # cd /home/tejas; \
    # cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
    # ./gcp_setup_stellar_private.sh;"'
    #     print(command)
    #     output = os.system(command)
    #     print(output)
    
    # # Execute in parallel like your example
    # results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
    # print(results)

    

    
    def run_stellar_private(i):


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        instance_name = f"tsm-sc-{i:03}"
        
        # ----------------------------------------------------------------------------------
        # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
        # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
        # '< /dev/null' ensures the process doesn't wait for input.
        # ----------------------------------------------------------------------------------
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
    > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
        
        # os.system should now return immediately because the remote shell exits
        output = os.system(command)
        print(f"Return code for {instance_name}: {output}")


        
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))




    # Corrected Loop (to run 0, 1, 2, 3)
    results = Parallel(n_jobs=48)(delayed(run_stellar_private)(i) for i in range(num_nodes))
    # results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in [3,2,1,0])
    

    # time.sleep(3)
    # for i in range(num_nodes):
    
        # run_stellar_private(num_nodes-i-1)
        # time.sleep(2)
    # run_stellar_private(0)
    print(results)
    print("All SSH commands executed. Nodes should be starting up in the background.")
    
    time.sleep(150)
    
    # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [1])
    # time.sleep(10)
    # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [2])
    # time.sleep(10)
    # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [3])
    # time.sleep(10)
    # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [4])
    # time.sleep(10)
    # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [5])
    # time.sleep(10)

    # time.sleep(50)
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "_v2" 
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "memory_"+ str(num_nodes) + "_no_cleanup_v2" 
    local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "test_"+ str(num_nodes) + "_refine" 

    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "_zone_" + str(zone_no)+"_v2" 
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "memory_" + str(num_nodes) + "_node_failure"
    
    # Ensure the local base destination directory exists
    os.makedirs(local_base_destination, exist_ok=True)
    
    
    def copy_folder_from_instance(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
            
        """
        Constructs and executes the gcloud compute scp command to copy a specific 
        nodeN folder from instance i to a local folder named after the instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        node_folder = f"node{node_number}"
    
        # 1. Define the specific REMOTE source path on the instance
        # Example: /home/tejas/stellar-private/node1
        remote_source_path = os.path.join(remote_base_folder, node_folder)
        
        # 2. Define the LOCAL destination path
        # We'll use the instance name for the subfolder to keep backups separate
        local_destination_path = os.path.join(local_base_destination, instance_name)
        os.makedirs(local_destination_path, exist_ok=True)
        
        # The SCp command requires the remote path to be formatted as:
        # [INSTANCE_NAME]:[REMOTE_SRC]
        remote_source = f"{instance_name}:{remote_source_path}"
        
        # The command reverses the source (remote) and destination (local)
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{remote_source}" "{local_destination_path}"'
    
        print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Copy from {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    # ---
    # Execute the copy operation in parallel
    # ---
    
    results = Parallel(n_jobs=48)(
        delayed(copy_folder_from_instance)(i) for i in range(3)
    )
    
    print("\n--- Summary of Download Results ---")
    print(results)
    

In [ ]:
    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

In [ ]:
# PROJECT=uni-ursa-major-tejas-lab
# ZONE=us-west1-b
# INSTANCE=tsm-sc-000
# IMAGE_FAMILY=tsm-sc-family
# gcloud compute images create ${IMAGE_FAMILY}-$(date +%Y%m%d-%H%M) --project=$PROJECT --source-disk=$INSTANCE --source-disk-zone=$ZONE  --family=$IMAGE_FAMILY --storage-location=us


In [ ]:

    # results = Parallel(n_jobs=48)(
    #     delayed(copy_folder_from_instance)(i) for i in [8]
    # )